## Task 3, part 4 - Modelling of the third model

This model predicts each perturbation's RNA log2FC fingerprint from that *same* perturbation's own effect on the 24 measured surface proteins. Unlike models 1 and 2, which only ever used information knowable without perturbing the gene, this uses a different readout (protein) of the same held-out perturbation as a test-time input -- explicitly sanctioned since perturbations are only held out from *training*, and may be used at test time.

In [1]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [2]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()

selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape

(150, 2042)

## Compute the protein log2FC "fingerprint"

Mirrors exactly what Task3_01 did for RNA: for each condition, compare the mean protein expression of each perturbation's cells to that condition's control cells, on a log2 scale with a pseudocount. 4 of the 24 measured "proteins" are isotype controls (antibody background-binding controls, not real markers) and are dropped, leaving 20 real surface markers.

In [3]:
protein = sc.read_h5ad(f"{DATA_DIR}/protein.h5ad")

# drop isotype controls (antibody background-binding controls, not real markers)
isotype_controls = ["Rat_IgG2a", "Mouse_IgG1", "Mouse_IgG2a", "Mouse_IgG2b"]
protein = protein[:, ~protein.var_names.isin(isotype_controls)].copy()

# keep only control cells and cells belonging to one of the 50 selected perturbations
relevant_mask = protein.obs["perturbation"].isin(selected_50) | (protein.obs["perturbation"] == "control")
protein = protein[relevant_mask.values].copy()

# normalize (no log-transform needed here, log2FC is computed on the normalized scale directly, as in Task3_01)
sc.pp.normalize_total(protein, target_sum=1e4)
protein_norm = np.asarray(protein.X.todense())

protein.shape

/tmp/ipykernel_45501/1216798519.py:12: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(protein, target_sum=1e4)


(94502, 20)

In [4]:
PSEUDOCOUNT = 1.0

# mean normalized protein expression of control cells, per condition
control_means_protein = {}
for cond in conditions:
    mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == "control")
    control_means_protein[cond] = protein_norm[mask.values].mean(axis=0)

# log2FC protein fingerprint per (perturbation, condition), relative to control cells of that condition
protein_FC = {}
for pert in selected_50:
    for cond in conditions:
        mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == pert)
        pert_mean = protein_norm[mask.values].mean(axis=0)
        protein_FC[(pert, cond)] = np.log2((pert_mean + PSEUDOCOUNT) / (control_means_protein[cond] + PSEUDOCOUNT))

protein_FC_index = pd.MultiIndex.from_tuples(protein_FC.keys(), names=["perturbation", "condition"])
protein_FC_df = pd.DataFrame(np.vstack(list(protein_FC.values())), index=protein_FC_index, columns=protein.var_names)

protein_FC_df.shape

(150, 20)

## Reduce the target: PCA on the training RNA fingerprints

Same approach as model 2: predicting all ~2042 RNA genes directly from only 20 protein features and 40 training genes is too many outputs for too little data, so fit PCA on the 40 training genes' RNA fingerprints (per condition) and predict position along those axes instead. This only uses training labels -- the 10 held-out genes are never involved in defining this space.

In [5]:
N_TARGET_PCS = 10

# per condition: fit PCA on the 40 training genes' RNA fingerprints, and store each training gene's score
target_pca_by_condition = {}
target_scores = {}  # (gene, condition) -> score along the target PCs, training genes only
for cond in conditions:
    train_fingerprints = np.vstack([pert_FC_selected.loc[(g, cond)].values for g in train_40])
    pca = PCA(n_components=N_TARGET_PCS, random_state=42)
    scores = pca.fit_transform(train_fingerprints)
    target_pca_by_condition[cond] = pca
    for gene, score in zip(train_40, scores):
        target_scores[(gene, cond)] = score

# how much of the training fingerprints' variance these 10 PCs capture, per condition
{cond: target_pca_by_condition[cond].explained_variance_ratio_.sum() for cond in conditions}

{'Control': np.float32(0.61914194),
 'IFNγ': np.float32(0.6695988),
 'Co-culture': np.float32(0.76921785)}

## Ridge regression prediction

For a query gene in a given condition: standardize its 20-dim protein log2FC vector, fit Ridge regression mapping protein log2FC -> the 10 RNA target-PC scores using the pool genes, predict the query's scores, then reconstruct the full RNA fingerprint with that condition's target PCA. The pool never includes the query gene itself, so this works for both leave-one-out cross-validation and predicting the actual held-out genes.

In [6]:
def ridge_predict(query_gene, condition, alpha, pool_genes):
    """Predict an RNA fingerprint via Ridge regression from protein log2FC to RNA target-PC scores."""
    # exclude the query gene itself from the pool used to fit the model
    fit_genes = [g for g in pool_genes if g != query_gene]

    X = np.vstack([protein_FC_df.loc[(g, condition)].values for g in fit_genes])
    y = np.vstack([target_scores[(g, condition)] for g in fit_genes])

    # standardize the protein features before regularized regression
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    ridge = Ridge(alpha=alpha)
    ridge.fit(X_scaled, y)

    query_X = scaler.transform(protein_FC_df.loc[(query_gene, condition)].values.reshape(1, -1))
    pred_scores = ridge.predict(query_X)

    # reconstruct the full RNA fingerprint from the predicted target-PC scores
    pred_fingerprint = target_pca_by_condition[condition].inverse_transform(pred_scores)[0]
    return pred_fingerprint

## Choosing the Ridge regularization strength (alpha) via leave-one-out cross-validation

Hold out one training gene at a time, predict it from the other 39 (per condition), and compare a few candidate alpha values. This never touches the 10 held-out test genes -- alpha is fixed before we ever look at them.

In [7]:
candidate_alphas = [0.1, 1, 10, 100, 1000, 10_000, 100_000, 1_000_000]

cv_mse_by_alpha = {}
for alpha in candidate_alphas:
    squared_errors = []
    for cond in conditions:
        for gene in train_40:
            # ridge_predict excludes the query gene itself from the pool, so this is a genuine leave-one-out prediction
            pred = ridge_predict(gene, cond, alpha, train_40)
            true = pert_FC_selected.loc[(gene, cond)].values
            squared_errors.append(np.mean((true - pred) ** 2))
    cv_mse_by_alpha[alpha] = np.mean(squared_errors)

best_alpha = min(cv_mse_by_alpha, key=cv_mse_by_alpha.get)
cv_mse_by_alpha, best_alpha

({0.1: np.float32(0.0070858756),
  1: np.float32(0.005488515),
  10: np.float32(0.003390293),
  100: np.float32(0.0023222358),
  1000: np.float32(0.0021853666),
  10000: np.float32(0.0021873454),
  100000: np.float32(0.0021882053),
  1000000: np.float32(0.0021882995)},
 1000)

## Predict the held-out test genes and evaluate

Use the chosen alpha to predict each of the 10 held-out genes' RNA fingerprint from their own protein log2FC, then evaluate with the same metrics used for the other models so results are directly comparable.

In [8]:
# predict each held-out (gene, condition) pair from the 40 training genes, using the CV-chosen alpha
ridge_predictions = {
    (gene, cond): ridge_predict(gene, cond, best_alpha, train_40)
    for cond in conditions
    for gene in test_10
}


def evaluate_predictions(true_df, predictions_by_row):
    """Compare each true fingerprint against its predicted fingerprint (looked up per row)."""
    records = []
    for (pert, cond), true_fc in true_df.iterrows():
        pred_fc = predictions_by_row[(pert, cond)]
        pearson_r, _ = pearsonr(true_fc, pred_fc)
        spearman_r, _ = spearmanr(true_fc, pred_fc)
        mse = np.mean((true_fc - pred_fc) ** 2)
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
        })
    return pd.DataFrame(records)


ridge_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], ridge_predictions)
ridge_eval

,perturbation,condition,pearson_r,spearman_r,mse
0,KCNN4,Control,0.828076,0.452951,0.001510
1,KCNN4,IFNγ,0.844792,0.393864,0.001099
2,KCNN4,Co-culture,0.865248,0.340129,0.001129
3,TIMM50,Control,0.740555,0.392573,0.002776
4,TIMM50,IFNγ,0.626393,0.295484,0.003330
5,TIMM50,Co-culture,0.700764,0.198442,0.003850
6,TXNDC17,Control,0.807620,0.494533,0.004335
7,TXNDC17,IFNγ,0.623309,0.439239,0.004303
8,TXNDC17,Co-culture,0.752614,0.365463,0.004496
9,CORO1A,Control,0.804729,0.374306,0.001203


In [9]:
metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition breakdown (n=10 genes each) -- for biological interpretation
per_condition = ridge_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled across all held-out (gene, condition) pairs (n=30) -- single headline number, comparable to the other models
overall = ridge_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.593808  0.498146   0.265838  0.105129  0.003968  0.005932
Control     0.755588  0.135749   0.399165  0.116829  0.002491  0.001575
IFNγ        0.719905  0.251020   0.377741  0.097322  0.002410  0.001890

In [10]:
overall

,pearson_r,spearman_r,mse
mean,0.689767,0.347582,0.002956
std,0.327519,0.118914,0.003651
